# Bronze to Silver Transformation (Incremental, Auto Loader)

Reads new Bronze files incrementally via Auto Loader, applies cleaning/
dedup/labeling logic per micro-batch and appends results to the Silver
Delta table.

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType,
    IntegerType, BooleanType, ArrayType
)
from pyspark.sql.functions import (
    from_json, col, explode, when, row_number, desc, sum as spark_sum
)
from pyspark.sql import Window

# Path to raw Bronze JSON files. both manually uploaded and auto ingested
# files land here
bronze_path = "/Volumes/logistics_pipeline/bronze/vessel_positions_raw/"

# Auto Loader's internal checkpoints. this tracks which files have already been
# read (read_checkpoint) and how far the write has progressed
# (write_checkpoint). These are Auto Loader's state, not data.
read_checkpoint_path = "/Volumes/logistics_pipeline/bronze/_checkpoints/bronze_to_silver_read"
write_checkpoint_path = "/Volumes/logistics_pipeline/bronze/_checkpoints/bronze_to_silver_write"

silver_table = "logistics_pipeline.silver.vessel_positions_clean"
silver_quarantine_table = "logistics_pipeline.silver.vessel_positions_quarantine"

# Explicit schema for one vessel record 
vessel_schema = StructType([
    StructField("mmsi", LongType(), True),
    StructField("imo", LongType(), True),
    StructField("vessel_name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("cog", DoubleType(), True),
    StructField("sog", DoubleType(), True),
    StructField("heading", IntegerType(), True),
    StructField("nav_status", IntegerType(), True),
    StructField("suspected_glitch", BooleanType(), True),
    StructField("timestamp", StringType(), True),            # cast to real timestamp 
    StructField("processed_timestamp", StringType(), True),  # cast to real timestamp 
])

## Read new Bronze files (Auto Loader) and parse the vessels array

In [0]:
# cloudFiles = Auto Loader's format. Only picks up files not yet seen,
# tracked via read_checkpoint_path. this is the core incremental behavior
df_bronze_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", read_checkpoint_path)
    .option("multiline", "true")   # each file is one JSON object, not newline delimited
    .load(bronze_path)
)

# from_json + our explicit vessel_schema fixes Auto Loader's inference
# flattening "vessels" into a raw string instead of a proper array/struct
df_bronze = df_bronze_raw.withColumn(
    "vessels_parsed",
    from_json(col("vessels"), ArrayType(vessel_schema))
)

## Explode nested vessel arrays into individual rows

In [0]:
# Turns "one row per file, containing an array" into "one row per vessel" -

df_exploded = df_bronze.select(explode("vessels_parsed").alias("vessel"))

## Process each micro-batch and append to Silver (+ quarantine)

In [0]:
def process_batch(batch_df, batch_id):
    # Auto Loader can trigger an empty batch (e.g. no new files this run) 
    # skip processing entirely rather than running empty transformations
    if batch_df.isEmpty():
        print(f"[BATCH {batch_id}] No new data this run.")
        return

    # Flatten: pull fields out of the vessel struct, enforce explicit types
    df_flat = batch_df.select(
        col("vessel.mmsi").cast("long").alias("mmsi"),
        col("vessel.imo").cast("long").alias("imo"),
        col("vessel.vessel_name").alias("vessel_name"),
        col("vessel.latitude").cast("double").alias("latitude"),
        col("vessel.longitude").cast("double").alias("longitude"),
        col("vessel.cog").cast("double").alias("cog"),
        col("vessel.sog").cast("double").alias("sog"),
        col("vessel.heading").cast("int").alias("heading"),
        col("vessel.nav_status").cast("int").alias("nav_status"),
        col("vessel.suspected_glitch").alias("suspected_glitch"),
        col("vessel.timestamp").cast("timestamp").alias("broadcast_timestamp"),
        col("vessel.processed_timestamp").cast("timestamp").alias("processed_timestamp"),
    )

    # Dedup: within THIS micro batch, keep only the most recently processed
    # version of each (mmsi, broadcast_timestamp) 
    window_spec = Window.partitionBy("mmsi", "broadcast_timestamp").orderBy(desc("processed_timestamp"))
    df_deduped = (
        df_flat
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)   # keep only the latest processed version
        .drop("row_num")               # remove other/helper column
    )

    # Nav status labels, this is standard AIS codes (ITU-R M.1371), not
    # VesselAPI specific
    df_labeled = df_deduped.withColumn(
        "nav_status_label",
        when(col("nav_status") == 0, "Under way using engine")
        .when(col("nav_status") == 1, "At anchor")
        .when(col("nav_status") == 2, "Not under command")
        .when(col("nav_status") == 3, "Restricted maneuverability")
        .when(col("nav_status") == 4, "Constrained by draught")
        .when(col("nav_status") == 5, "Moored")
        .when(col("nav_status") == 6, "Aground")
        .when(col("nav_status") == 7, "Engaged in fishing")
        .when(col("nav_status") == 8, "Under way sailing")
        .when(col("nav_status").isNull(), "Not reported")
        .otherwise("Other/reserved code")
    )

    # Spliting into clean vs. quarantined. that is VesselAPI's own glitch flag
    df_clean = df_labeled.filter(col("suspected_glitch") == False)
    df_quarantine = df_labeled.filter(col("suspected_glitch") == True)

    # Append (not overwrite) with this, each micro-batch adds new rows without
    # discarding previously processed history
    df_clean.write.mode("append").saveAsTable(silver_table)

    # Quarantined records are preserved too, not silently dropped, for
    # production quality.
    if not df_quarantine.isEmpty():
        df_quarantine.write.mode("append").saveAsTable(silver_quarantine_table)

    print(f"[BATCH {batch_id}] Appended {df_clean.count()} clean rows to {silver_table}, "
          f"{df_quarantine.count()} quarantined rows to {silver_quarantine_table}")


# trigger(availableNow=True): process whatever new files exist right now,
# then stop and not a continuously running stream
query = (
    df_exploded.writeStream
    .foreachBatch(process_batch)
    .trigger(availableNow=True)
    .option("checkpointLocation", write_checkpoint_path)  # tracks write progress, separate from the read checkpoint
    .start()
)

query.awaitTermination()

Null handling  (against the persisted Silver table)

Runs after the streaming write completes, querying the real, currently
persisted Silver table directly, since there's no single in memory
DataFrame representing "all of Silver" in this incremental structure.

In [0]:
# Read the actual persisted table, not an in memory variable, since data
# now arrives incrementally across many micro batches rather than one
# single DataFrame
df_silver_current = spark.table(silver_table)
total_rows = df_silver_current.count()

# Same null counting for current
# real data
null_counts = df_silver_current.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver_current.columns
])

display(null_counts)
print(f"Total rows currently in Silver: {total_rows}")